In [22]:
'''
Names: Prashant Ram, Muhammad Hassan, Gandhar Tare, Mohamed Burhan
Course: CSC 177
Assignment: 3 - Linear Regression
Professor: Jagannadha Chidella
Details: 
'''

import pandas
import matplotlib.pyplot as matPlot
import seaborn
import numpy
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn import linear_model
from sklearn.model_selection import train_test_split



In [23]:
'''
This block reads the data from a .csv file and makes columns for the data
'''
try:
    # import the data from the .csv file. "data" = dataframe. Our data is using ; as a separator and quotes to enclose
    # the data. The first row is the header row so we use the correct separator and quote character or else pandas treats these
    # data as a single column which we don't want.
    data = pandas.read_csv('Churn_Modelling.csv')

    #print the data attribute columns
    print(data.columns)
        
    # print the number of rows and columns
    print(f'We have {data.shape[0]} rows')
    print(f'We have {data.shape[1]} columns')

    # print the first 5 rows of the data
    print('\n', data.head())
        
except Exception as e:
    print("An error occured while trying to read from the CSV file", e)

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')
We have 10000 rows
We have 14 columns

    RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00      

In [24]:
'''
This block counts the number of missing values in each column
'''

#list to hold the columns with missing values
missingColumns = []

#our original number of rows with unclean data
original_rows = data.shape[0]

try:
    #loop through the columns and get the number of missing values
    for i in data.columns:
        #get the total of column of missing values
        total = data[i].isnull().sum()
        #display the column and the missing values number
        print(f"{i}: {total}")

        #add missing value columns to the list
        if total > 0:
            missingColumns.append(i)

    #finally display which columns had missing values
    if len(missingColumns) == 0:
        print("\nThere were no missing values in the data set.")
    else:
        #display the columns with missing values
        print(f"\nThese columns had missing data: {missingColumns}")
    
#catch any reading errors
except Exception as e:
    print("umm. An error occurred while trying to count the missing values..", e)

RowNumber: 0
CustomerId: 0
Surname: 0
CreditScore: 0
Geography: 0
Gender: 0
Age: 0
Tenure: 0
Balance: 0
NumOfProducts: 0
HasCrCard: 0
IsActiveMember: 0
EstimatedSalary: 0
Exited: 0

There were no missing values in the data set.


In [25]:
'''
This block drops all rows and columns with missing data
'''
# print the number of rows and columns
original_rows = data.shape[0]
original_columns = data.shape[1]
print(f'We have {original_rows} [original] rows and {original_columns} columns')
print('\nOriginal Data:', data.head())
    
#drops all rows with no data
data = data.dropna()

clean_rows = data.shape[0]
clean_columns = data.shape[1]
print(f'\nWe have {clean_rows} [Clean] rows and {clean_columns} columns')
print('\nCleaned data Data:', data.head())

print(f'\n\nWe had {original_rows - clean_rows} rows and {original_columns - clean_columns} columns of missing data')

We have 10000 [original] rows and 14 columns

Original Data:    RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        1

In [26]:
'''This block checks for duplicate rows and drops the ones that are duplicates'''
total_duplicates = data.duplicated().sum()

if total_duplicates != 0:
    data = data.drop_duplicates(inkeep='first')
    print(f"\nThere were {total_duplicates} duplicate rows in the data set.")
else:
    print("\nThere were no duplicate rows in the data set.")



There were no duplicate rows in the data set.


In [27]:
'''
All the data columns are numerical so we do not need to do any conversion from categorical to numerical data. To verfiy, we will
print out the data types.
'''
data_types  = data.dtypes
print(data_types)

RowNumber            int64
CustomerId           int64
Surname             object
CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object


In [ ]:
'''This block drops all columns which do not help the model learn'''
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
print(data.columns)


Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='object')


In [31]:
'''This codes will remove any outliers in the data using z-score method. Out threshold is 3.0. Any data point with a z-score greater than 3.0 will 
be removed.'''

columns = data.select_dtypes(include=['float64', 'int64']).columns
with_outliers = data

#run this 5 times because for some reason it's unable to remove all the outliers in 3 loops.
for i in range(0, 5):
    for col in columns:
        Q1 = with_outliers[col].quantile(0.25)    
        Q3 = with_outliers[col].quantile(0.75)     
        IQR = Q3 - Q1                         
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        with_outliers= with_outliers[(with_outliers[col] >= lower_bound) & (with_outliers[col] <= upper_bound)]
        data = with_outliers

print("Original dataset with outliers:", original_rows)
print("Cleaned dataset without outliers:", with_outliers.shape[0])
print("Number of rows with outliers removed:", original_rows - with_outliers.shape[0])


Original dataset with outliers: 10000
Cleaned dataset without outliers: 7390
Number of rows with outliers removed: 2610


In [32]:
'''This block converts all the categorical data to numerical data using one-hot encoding.'''

data['Geography'] = data['Geography'].replace({'France': 2, 'Spain': 1, 'Germany':0}).infer_objects(copy=False)
data['Gender'] = data['Gender'].replace({'Female': 1, 'Male': 0}).infer_objects(copy=False)

C:\Users\rampr\AppData\Local\Temp\ipykernel_29964\1481568574.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Geography'] = data['Geography'].replace({'France': 2, 'Spain': 1, 'Germany':0}).infer_objects(copy=False)
C:\Users\rampr\AppData\Local\Temp\ipykernel_29964\1481568574.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Gender'] = data['Gender'].replace({'Female': 1, 'Male': 0}).infer_objects(copy=False)


In [34]:
'''Save the cleaned dataset to a new csv file. This will be used for the linear regression model.'''
try:
    data.to_csv('processed_Churn_Data.csv', index=False)
    print("Cleaned data saved to current directory as file name = processed_Churn_Data.csv")
except Exception as e:
    print("An error occurred while trying to save the cleaned data to a CSV file", e)

Cleaned data saved to current directory as file name = processed_Churn_Data.csv
